# ZFN Dev-Image Experiment Series (GHCR)

Stepwise notebook to design and evaluate ZFN workflow experiments using the published container image.

This notebook implements the series:
- **E0** image provenance gate
- **E1** smoke test
- **E2** algorithm comparison
- **E3** stringency sweep
- **E4** constraint-ablation study
- **E5** robustness/negative controls

> Designed for manual, stepwise execution. Most command cells are gated by booleans so you decide what to run.

## 1) Environment + Global Configuration

In [ ]:
from __future__ import annotations

import csv
import json
import os
import shlex
import shutil
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import pandas as pd

# ---- Path discovery ----
NOTEBOOK_DIR = Path.cwd().resolve()


def find_repo_root(start: Path) -> Path:
    """Walk upward from a starting directory until a workspace root is found."""
    current = start
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"Could not find repo root from: {start}")


HOST_WORKSPACE = find_repo_root(NOTEBOOK_DIR)
CONTAINER_WORKSPACE = Path("/workspace")

# ---- User controls ----
LOCAL_IMAGE = "sirnaforge:latest"
IMAGE = os.environ.get("SIRNAFORGE_IMAGE", LOCAL_IMAGE)
BUILD_LOCAL_IMAGE = True  # run `make docker-build` in Section 2A before experiments
RUN_ROOT = NOTEBOOK_DIR / "zfn_experiment_runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Core budget for workflow execution; all workflow runs use this unless overridden.
ALL_CORES = max(1, os.cpu_count() or 1)

# Toggle execution behavior
ALLOW_DOCKER_RUN = True  # set True only when ready to execute docker commands
STOP_ON_ERROR = False  # keep running full matrix, even if negative controls fail
DEBUG_FIRST_FAILURE = True  # print stdout/stderr tail on first failure
PRINT_LOG_TAIL_LINES = 120

# Common CCR5 constants from repo integration tests
CCR5_LEFT_HALF_SITE = "GTCATCCTCATC"
CCR5_RIGHT_HALF_SITE = "AAACTGCAAAAG"

print("Notebook dir:", NOTEBOOK_DIR)
print("Host workspace:", HOST_WORKSPACE)
print("Container workspace:", CONTAINER_WORKSPACE)
print("Image:", IMAGE)
print("Build local image:", BUILD_LOCAL_IMAGE)
print("Run root:", RUN_ROOT)
print("ALL_CORES (default workflow cores):", ALL_CORES)
print("ALLOW_DOCKER_RUN:", ALLOW_DOCKER_RUN)
print("STOP_ON_ERROR:", STOP_ON_ERROR)

In [ ]:
# ---- 1A) Build local Docker image from Makefile ----
if BUILD_LOCAL_IMAGE:
    build_cmd = ["make", "docker-build"]
    print("$", " ".join(build_cmd))
    if ALLOW_DOCKER_RUN:
        build_proc = subprocess.run(
            build_cmd,
            cwd=HOST_WORKSPACE,
            check=False,
            text=True,
            capture_output=True,
        )
        print("rc=", build_proc.returncode)
        if build_proc.stdout:
            print(build_proc.stdout[-4000:])
        if build_proc.stderr:
            print(build_proc.stderr[-4000:])
        if build_proc.returncode != 0:
            raise RuntimeError("make docker-build failed")
    else:
        print("Skipped: set ALLOW_DOCKER_RUN=True to execute make docker-build.")
else:
    print("Skipping local image build (BUILD_LOCAL_IMAGE=False).")

## 2) Helpers

In [ ]:
from typing import TypedDict, cast


@dataclass
class CmdResult:
    """Captured result for a shell command executed during an experiment."""

    name: str
    cmd: str
    returncode: int
    elapsed_s: float
    stdout_path: str
    stderr_path: str
    output_dir: str | None


class RunSummary(TypedDict):
    """Normalized summary payload for one workflow output directory."""

    output_dir: str
    offtarget_rows: int | None
    candidate_rows: int | None
    top_composite_score: float | None
    workflow_mode: str | None
    left_half_site: str | None
    right_half_site: str | None


def run_shell(name: str, cmd: str, run_dir: Path, output_dir: Path | None = None) -> CmdResult:
    """Execute one shell command and persist stdout/stderr logs."""
    run_dir.mkdir(parents=True, exist_ok=True)
    stdout_path = run_dir / f"{name}.stdout.log"
    stderr_path = run_dir / f"{name}.stderr.log"

    start = time.time()
    proc = subprocess.run(cmd, check=False, shell=True, text=True, capture_output=True)
    elapsed = time.time() - start

    stdout_path.write_text(proc.stdout, encoding="utf-8")
    stderr_path.write_text(proc.stderr, encoding="utf-8")

    return CmdResult(
        name=name,
        cmd=cmd,
        returncode=proc.returncode,
        elapsed_s=elapsed,
        stdout_path=str(stdout_path),
        stderr_path=str(stderr_path),
        output_dir=str(output_dir) if output_dir else None,
    )


def to_container_path(host_path: Path) -> Path:
    """Map a host path into the mounted container workspace path."""
    host_resolved = host_path.resolve()
    rel = host_resolved.relative_to(HOST_WORKSPACE)
    return CONTAINER_WORKSPACE / rel


def print_log_tail(log_path: str, n_lines: int = 120) -> None:
    """Print a bounded tail for a log file if it exists."""
    p = Path(log_path)
    if not p.exists():
        print(f"[missing] {p}")
        return
    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    tail = lines[-n_lines:] if n_lines > 0 else lines
    print(f"--- {p} (last {len(tail)} lines) ---")
    for line in tail:
        print(line)


def debug_failed_result(result: CmdResult, n_lines: int = 120) -> None:
    """Print command and log tails for a failed command result."""
    print(f"\nFailure in {result.name} (rc={result.returncode})")
    print("Command:", result.cmd)
    print_log_tail(result.stdout_path, n_lines=n_lines)
    print_log_tail(result.stderr_path, n_lines=n_lines)


def docker_workflow_cmd(
    run_id: str,
    output_dir: Path,
    search_space_fasta: Path | None = None,
    search_space_override: str | None = None,
    algorithm: str = "zfn_v2",
    spacer_lengths: str = "5,6",
    max_mismatches: int = 4,
    extra_args: list[str] | None = None,
) -> str:
    """Build a dockerized `sirnaforge workflow` command string."""
    output_dir_host = output_dir.resolve()
    output_dir_host.parent.mkdir(parents=True, exist_ok=True)

    if search_space_override:
        zfn_search_space_arg = search_space_override
    else:
        if search_space_fasta is None:
            raise ValueError("search_space_fasta is required when search_space_override is not set")
        search_space_host = search_space_fasta.resolve()
        zfn_search_space_arg = str(to_container_path(search_space_host))

    output_dir_container = to_container_path(output_dir_host)

    base = [
        "docker run --rm",
        f"-v {shlex.quote(str(HOST_WORKSPACE))}:{shlex.quote(str(CONTAINER_WORKSPACE))}",
        f"-w {shlex.quote(str(CONTAINER_WORKSPACE))}",
        shlex.quote(IMAGE),
        "sirnaforge workflow",
        shlex.quote(run_id),
        "--design-mode zfn",
        f"--zfn-left-half-site {shlex.quote(CCR5_LEFT_HALF_SITE)}",
        f"--zfn-right-half-site {shlex.quote(CCR5_RIGHT_HALF_SITE)}",
        f"--zfn-search-space {shlex.quote(zfn_search_space_arg)}",
        f"--zfn-spacer-lengths {shlex.quote(spacer_lengths)}",
        f"--zfn-max-mismatches {max_mismatches}",
        f"--zfn-algorithm {shlex.quote(algorithm)}",
        f"--output-dir {shlex.quote(str(output_dir_container))}",
    ]
    if extra_args:
        base.extend(extra_args)
    return " ".join(base)


def summarize_single_run(output_dir: Path) -> RunSummary:
    """Read workflow outputs and return a typed summary dict."""
    summary: RunSummary = {
        "output_dir": str(output_dir),
        "offtarget_rows": None,
        "candidate_rows": None,
        "top_composite_score": None,
        "workflow_mode": None,
        "left_half_site": None,
        "right_half_site": None,
    }

    ot_csv = output_dir / "sirnaforge" / "zfn_offtarget_sites.csv"
    cand_json = output_dir / "sirnaforge" / "zfn_candidate_summary.json"
    wf_json = output_dir / "logs" / "workflow_summary.json"

    if ot_csv.exists():
        with ot_csv.open(newline="", encoding="utf-8") as fh:
            summary["offtarget_rows"] = sum(1 for _ in csv.DictReader(fh))

    if cand_json.exists():
        raw_data = json.loads(cand_json.read_text(encoding="utf-8"))
        data = cast(dict[str, Any], raw_data) if isinstance(raw_data, dict) else {}
        candidates_raw = data.get("candidates", [])
        candidates = candidates_raw if isinstance(candidates_raw, list) else []
        summary["candidate_rows"] = len(candidates)
        if candidates and isinstance(candidates[0], dict):
            top = cast(dict[str, Any], candidates[0]).get("composite_score")
            if isinstance(top, (int, float)):
                summary["top_composite_score"] = float(top)

    if wf_json.exists():
        raw_wf = json.loads(wf_json.read_text(encoding="utf-8"))
        wf = cast(dict[str, Any], raw_wf) if isinstance(raw_wf, dict) else {}
        mode = wf.get("workflow_mode")
        left = wf.get("left_half_site")
        right = wf.get("right_half_site")
        summary["workflow_mode"] = mode if isinstance(mode, str) else None
        summary["left_half_site"] = left if isinstance(left, str) else None
        summary["right_half_site"] = right if isinstance(right, str) else None

    return summary

## 2B) Parallel Profiling Helpers

> Optional profiling utilities for tuning `--cores` and internal ZFN stride behavior on memory-limited hosts.

In [ ]:
import re


def _extract_zfn_phase_timings(log_path: str) -> dict[str, float]:
    """Extract the most recent JSON phase timing payload from a command log."""
    payload: dict[str, float] = {}
    p = Path(log_path)
    if not p.exists():
        return payload
    for line in p.read_text(encoding="utf-8", errors="replace").splitlines():
        marker = "ZFN search phase timings:"
        if marker not in line:
            continue
        raw = line.split(marker, maxsplit=1)[1].strip()
        # Handle plain JSON payloads and logger-prefixed lines.
        m = re.search(r"\{.*\}$", raw)
        candidate = m.group(0) if m else raw
        try:
            decoded = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if isinstance(decoded, dict):
            payload = {k: float(v) for k, v in decoded.items() if isinstance(v, (int, float))}
    return payload


def _inject_runtime_tuning(cmd: str, cores: int, stride: int) -> str:
    """Inject env vars for internal tuning without changing CLI UX."""
    prefix = f"docker run --rm -e SIRNAFORGE_CORES={cores} -e SIRNAFORGE_ZFN_WINDOW_STRIDE={stride}"
    return cmd.replace("docker run --rm", prefix, 1)


def profile_parallel_matrix(
    *,
    base_run_id: str,
    search_space_fasta: Path,
    cores_values: list[int],
    stride_values: list[int],
    repeats: int = 1,
    algorithm: str = "zfn_v2",
    spacer_lengths: str = "5,6",
    max_mismatches: int = 4,
) -> pd.DataFrame:
    """Run a matrix sweep over cores/stride and return one tidy dataframe."""
    rows: list[dict[str, Any]] = []

    for cores in cores_values:
        for stride in stride_values:
            for rep in range(1, repeats + 1):
                profile_run_id = f"{base_run_id}_C{cores}_S{stride}_R{rep}"
                output_dir = RUN_ROOT / "E2B_parallel_profile" / profile_run_id

                cmd = docker_workflow_cmd(
                    run_id=profile_run_id,
                    output_dir=output_dir,
                    search_space_fasta=search_space_fasta,
                    search_space_override=None,
                    algorithm=algorithm,
                    spacer_lengths=spacer_lengths,
                    max_mismatches=max_mismatches,
                    extra_args=[],
                )
                cmd = _inject_runtime_tuning(cmd, cores=cores, stride=stride)

                result = run_shell(
                    name=f"E2B__{profile_run_id}",
                    cmd=cmd,
                    run_dir=RUN_ROOT / "_logs",
                    output_dir=output_dir,
                )

                summary = summarize_single_run(output_dir)
                phase = _extract_zfn_phase_timings(result.stdout_path)
                if not phase:
                    phase = _extract_zfn_phase_timings(result.stderr_path)

                row: dict[str, Any] = {
                    "run_id": profile_run_id,
                    "cores": cores,
                    "stride": stride,
                    "repeat": rep,
                    "returncode": result.returncode,
                    "elapsed_s": result.elapsed_s,
                    "offtarget_rows": summary["offtarget_rows"],
                    "candidate_rows": summary["candidate_rows"],
                    "top_composite_score": summary["top_composite_score"],
                    "search_shards_s": phase.get("search_shards_s"),
                    "rank_s": phase.get("rank_s"),
                    "dedupe_s": phase.get("dedupe_s"),
                    "stdout_log": result.stdout_path,
                    "stderr_log": result.stderr_path,
                    "output_dir": str(output_dir),
                }
                rows.append(row)

    return pd.DataFrame(rows)

## 3) Build Synthetic Search Space from Checked-in CCR5 Fixtures

This mirrors the container integration test strategy and avoids requiring a full genome for initial experiments.

In [ ]:
from Bio.Seq import Seq

fixture_csv = HOST_WORKSPACE / "tests" / "unit" / "data" / "zfn" / "ccr5_s10_visible_rows.csv"

if not fixture_csv.exists():
    raise FileNotFoundError(f"Could not find fixture: {fixture_csv}")

with fixture_csv.open(newline="", encoding="utf-8") as fh:
    rows = list(csv.DictReader(fh))


def _row(gene: str) -> dict[str, str]:
    """Return one fixture row by closest gene symbol."""
    return next(r for r in rows if r.get("Closest gene") == gene)


def site_seq(plus: str, minus: str, spacer_len: int = 5) -> str:
    """Build plus-strand site sequence from half-sites and spacer length."""
    return plus.upper() + ("A" * spacer_len) + str(Seq(minus.upper()).reverse_complement())


ccr5_row = _row("CCR5")
csnk1g3_row = _row("CSNK1G3")
tmod1_row = _row("TMOD1")

sites = [
    site_seq(ccr5_row["(+) half-site"], ccr5_row["(−) half-site"], 5),
    site_seq(csnk1g3_row["(+) half-site"], csnk1g3_row["(−) half-site"], 5),
    site_seq(tmod1_row["(+) half-site"], tmod1_row["(−) half-site"], 5),
]

search_space = RUN_ROOT / "inputs" / "ccr5_synthetic_genome.fa"
search_space.parent.mkdir(parents=True, exist_ok=True)

genome_seq = "N" * 50 + ("N" * 100).join(sites) + "N" * 50
search_space.write_text(f">chr_synthetic\n{genome_seq}\n", encoding="utf-8")

print("Synthetic FASTA:", search_space)
print("Length:", len(genome_seq))

## 5) Build Experiment Matrix (E1–E6)

Creates a concrete run list you can inspect before executing anything.

- `E6_real_chr3` is the recommended real-data scale-up test (Ensembl human chromosome 3).
- `E6_real_hg38` remains optional as a full-primary-assembly stress test (likely to hit memory limits with current exhaustive implementation).
- Real-data runs apply a minimal shard override via `SIRNAFORGE_ZFN_SHARDING_JSON` to reduce memory pressure without changing alignment/mapping behavior.

In [ ]:
records: list[dict[str, Any]] = []

# E1: smoke
records.append(
    {
        "exp": "E1_smoke",
        "run_id": "CCR5_ZFN_SMOKE",
        "algorithm": "zfn_v2",
        "spacer_lengths": "5,6",
        "max_mismatches": 4,
        "search_space_override": None,
        "extra_args": [],
    }
)

# E2: algorithm comparison
for algo in ["zfn_v2", "homology"]:
    records.append(
        {
            "exp": "E2_algorithm",
            "run_id": f"CCR5_ZFN_ALGO_{algo.upper()}",
            "algorithm": algo,
            "spacer_lengths": "5,6",
            "max_mismatches": 4,
            "search_space_override": None,
            "extra_args": [],
        }
    )

# E3: stringency sweep
for mm in [2, 3, 4]:
    for sp in ["5", "5,6", "5,6,7"]:
        records.append(
            {
                "exp": "E3_stringency",
                "run_id": f"CCR5_ZFN_STR_MM{mm}_SP{sp.replace(',', '_')}",
                "algorithm": "zfn_v2",
                "spacer_lengths": sp,
                "max_mismatches": mm,
                "search_space_override": None,
                "extra_args": [],
            }
        )

# E4: constraint ablation
records.extend(
    [
        {
            "exp": "E4_constraints",
            "run_id": "CCR5_ZFN_CONSTRAINT_BASE",
            "algorithm": "zfn_v2",
            "spacer_lengths": "5,6",
            "max_mismatches": 4,
            "search_space_override": None,
            "extra_args": [],
        },
        {
            "exp": "E4_constraints",
            "run_id": "CCR5_ZFN_CONSTRAINT_GLOBAL",
            "algorithm": "zfn_v2",
            "spacer_lengths": "5,6",
            "max_mismatches": 4,
            "search_space_override": None,
            "extra_args": ["--zfn-max-mismatches-per-subfinger 1", "--zfn-subfinger-mutation overall:3:substitution"],
        },
        {
            "exp": "E4_constraints",
            "run_id": "CCR5_ZFN_CONSTRAINT_SUBFINGER",
            "algorithm": "zfn_v2",
            "spacer_lengths": "5,6",
            "max_mismatches": 4,
            "search_space_override": None,
            "extra_args": [
                "--zfn-max-mismatches-per-subfinger 1",
                "--zfn-subfinger-mutation overall:3:substitution",
                "--zfn-subfinger-mutation 3:2:transition,transversion",
            ],
        },
    ]
)

# E6 (recommended): real Ensembl chromosome 3
records.append(
    {
        "exp": "E6_real_chr3",
        "run_id": "CCR5_ZFN_REAL_CHR3",
        "algorithm": "zfn_v2",
        "spacer_lengths": "5,6",
        "max_mismatches": 3,
        "search_space_override": "https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.3.fa.gz",
        "extra_args": [
            f"--cores {ALL_CORES}",
            "--zfn-annotation https://ftp.ensembl.org/pub/current_gtf/homo_sapiens/Homo_sapiens.GRCh38.115.gtf.gz",
        ],
    }
)

# E6 (optional stress test): full Ensembl primary assembly
records.append(
    {
        "exp": "E6_real_hg38",
        "run_id": "CCR5_ZFN_REAL_HG38_PRIMARY",
        "algorithm": "zfn_v2",
        "spacer_lengths": "5,6",
        "max_mismatches": 2,
        "search_space_override": "ensembl_human_hg38_primary",
        "extra_args": [f"--cores {ALL_CORES}"],
    }
)

for record in records:
    record["output_dir"] = str((RUN_ROOT / str(record["exp"]) / str(record["run_id"])).resolve())

matrix = pd.DataFrame(records)
matrix

## 5B) Profile Cores/Stride Matrix (16GB-friendly)

> Runs a small profiling sweep to identify stable settings before expensive real-data runs.

In [ ]:
# Recommended small sweep for 16GB machines: keep cores conservative and test stride impact.
profile_cores = [1, 2, 3]
profile_strides = [1, 2]
profile_repeats = 1

print("ALLOW_DOCKER_RUN:", ALLOW_DOCKER_RUN)
print("Cores:", profile_cores, "Strides:", profile_strides)

if not ALLOW_DOCKER_RUN:
    print("Skipped profiling run. Set ALLOW_DOCKER_RUN=True to execute profiling matrix.")
else:
    prof_df = profile_parallel_matrix(
        base_run_id="CCR5_ZFN_PROFILE",
        search_space_fasta=search_space,
        cores_values=profile_cores,
        stride_values=profile_strides,
        repeats=profile_repeats,
        algorithm="zfn_v2",
        spacer_lengths="5,6",
        max_mismatches=4,
    )

    results_path = RUN_ROOT / "profile_parallel_matrix.csv"
    prof_df.to_csv(results_path, index=False)
    print("Saved:", results_path)

    summary_cols = [
        "cores",
        "stride",
        "returncode",
        "elapsed_s",
        "search_shards_s",
        "offtarget_rows",
        "candidate_rows",
    ]

    summary_df = (
        prof_df[summary_cols]
        .groupby(["cores", "stride", "returncode"], dropna=False, as_index=False)
        .agg(
            median_elapsed_s=("elapsed_s", "median"),
            median_search_shards_s=("search_shards_s", "median"),
            median_offtarget_rows=("offtarget_rows", "median"),
            median_candidate_rows=("candidate_rows", "median"),
        )
        .sort_values(["stride", "median_elapsed_s", "cores"])
    )

    print("\nProfile summary (lower elapsed is better; keep returncode==0):")
    display(summary_df)

    valid = summary_df[summary_df["returncode"] == 0]
    if not valid.empty:
        best = valid.iloc[0]
        print(
            "Suggested setting for this host:",
            f"cores={int(best['cores'])}",
            f"stride={int(best['stride'])}",
        )
    else:
        print("No successful profile run found. Check logs in RUN_ROOT / '_logs'.")

## 6) Preview Commands (Dry Run)

In [ ]:
preview_cmds: list[dict[str, str]] = []
for row in records:
    run_id = str(row["run_id"])
    exp = str(row["exp"])
    output_dir = Path(str(row["output_dir"]))
    search_override = row.get("search_space_override")
    search_space_override = str(search_override) if isinstance(search_override, str) else None
    algorithm = str(row["algorithm"])
    spacer_lengths = str(row["spacer_lengths"])
    max_mismatches = int(row["max_mismatches"])
    raw_extra_args = row.get("extra_args", [])
    extra_args = [str(arg) for arg in raw_extra_args] if isinstance(raw_extra_args, list) else []
    if not any(arg.startswith("--cores ") for arg in extra_args):
        extra_args.append(f"--cores {ALL_CORES}")
    cmd = docker_workflow_cmd(
        run_id=run_id,
        output_dir=output_dir,
        search_space_fasta=search_space,
        search_space_override=search_space_override,
        algorithm=algorithm,
        spacer_lengths=spacer_lengths,
        max_mismatches=max_mismatches,
        extra_args=extra_args,
    )
    preview_cmds.append({"exp": exp, "run_id": run_id, "cmd": cmd})

preview_df = pd.DataFrame(preview_cmds)
pd.set_option("display.max_colwidth", 180)
preview_df

## 7) Execute Selected Experiments

Set `selected_experiments` to a subset (for example `['E1_smoke', 'E2_algorithm']`) for staged runs.

Default below runs the full experiment set (E1–E6, including `E6_real_hg38`) and continues even if negative controls fail.

In [ ]:
selected_experiments = sorted({str(row["exp"]) for row in records})
run_manifest_path = RUN_ROOT / "run_manifest.jsonl"
results_csv_path = RUN_ROOT / "results_summary.csv"
run_records: list[dict[str, Any]] = []
failed_run: dict[str, Any] | None = None

if run_manifest_path.exists():
    run_manifest_path.unlink()
if results_csv_path.exists():
    results_csv_path.unlink()

for row in records:
    exp = str(row["exp"])
    if exp not in selected_experiments:
        continue

    run_id = str(row["run_id"])
    output_dir = Path(str(row["output_dir"]))
    search_override = row.get("search_space_override")
    search_space_override = str(search_override) if isinstance(search_override, str) else None
    algorithm = str(row["algorithm"])
    spacer_lengths = str(row["spacer_lengths"])
    max_mismatches = int(row["max_mismatches"])
    raw_extra_args = row.get("extra_args", [])
    extra_args = [str(arg) for arg in raw_extra_args] if isinstance(raw_extra_args, list) else []
    if not any(arg.startswith("--cores ") for arg in extra_args):
        extra_args.append(f"--cores {ALL_CORES}")

    run_log_dir = RUN_ROOT / "_logs"
    run_log_dir.mkdir(parents=True, exist_ok=True)
    for suffix in ("stdout", "stderr"):
        old_log = run_log_dir / f"{exp}__{run_id}.{suffix}.log"
        if old_log.exists():
            old_log.unlink()
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = docker_workflow_cmd(
        run_id=run_id,
        output_dir=output_dir,
        search_space_fasta=search_space,
        search_space_override=search_space_override,
        algorithm=algorithm,
        spacer_lengths=spacer_lengths,
        max_mismatches=max_mismatches,
        extra_args=extra_args,
    )

    print(f"\n[{exp}] {run_id}")
    print("$", cmd)

    if not ALLOW_DOCKER_RUN:
        print("Skipped: set ALLOW_DOCKER_RUN=True to execute.")
        continue

    result = run_shell(
        name=f"{exp}__{run_id}",
        cmd=cmd,
        run_dir=RUN_ROOT / "_logs",
        output_dir=output_dir,
    )

    run_rec = asdict(result)
    run_rec.update({"exp": exp, "run_id": run_id})
    run_rec.update(summarize_single_run(output_dir))
    run_records.append(run_rec)

    with run_manifest_path.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(run_rec) + "\n")

    if result.returncode != 0 and failed_run is None:
        failed_run = run_rec
        if DEBUG_FIRST_FAILURE:
            debug_failed_result(result, n_lines=PRINT_LOG_TAIL_LINES)
        if STOP_ON_ERROR:
            print(f"\nStopped after first failure: {exp}/{run_id}")
            break

results_df = pd.DataFrame(run_records)
if not results_df.empty:
    results_df.to_csv(results_csv_path, index=False)
    print("Saved:", results_csv_path)

if failed_run is not None:
    print("First failing run summary:")
    print(json.dumps(failed_run, indent=2))

results_df

## 8) Load Existing Results (if runs were done previously)

In [ ]:
results_csv = RUN_ROOT / "results_summary.csv"
if results_csv.exists():
    res = pd.read_csv(results_csv)
    print("Loaded:", results_csv)
    display(res.head(20))
else:
    print("No results_summary.csv yet. Run Section 7 first.")

## 9) Quick Analysis Views

In [ ]:
def _count_failures(series: pd.Series) -> int:
    """Count non-zero return codes in one group."""
    return int((series != 0).sum())


if (RUN_ROOT / "results_summary.csv").exists():
    res = pd.read_csv(RUN_ROOT / "results_summary.csv")

    print("By experiment:")
    by_experiment = (
        res.groupby("exp", dropna=False)
        .agg(
            n_runs=("run_id", "count"),
            n_fail=("returncode", _count_failures),
            median_elapsed_s=("elapsed_s", "median"),
            median_offtarget_rows=("offtarget_rows", "median"),
            median_top_score=("top_composite_score", "median"),
        )
        .reset_index()
    )
    display(by_experiment)

    print("E2 algorithm comparison:")
    e2 = res[res["exp"] == "E2_algorithm"][
        ["run_id", "returncode", "elapsed_s", "offtarget_rows", "top_composite_score"]
    ]
    display(e2.sort_values("run_id"))

    print("E3 stringency trend:")
    e3 = res[res["exp"] == "E3_stringency"][
        ["run_id", "returncode", "elapsed_s", "offtarget_rows", "top_composite_score"]
    ]
    display(e3.sort_values("run_id"))
else:
    print("No results to analyze yet.")

## 10) Reporting Parity Check (Toy vs Real HG38)

Compares key ZFN output schemas between the toy smoke run (`E1_smoke`) and real-scale run (`E6_real_hg38`).

Run this after executing Section 7 for `E6_real_hg38`.

In [ ]:
toy_dir = RUN_ROOT / "E1_smoke" / "CCR5_ZFN_SMOKE"
real_dir = RUN_ROOT / "E6_real_hg38" / "CCR5_ZFN_REAL_HG38_PRIMARY"


def _read_csv_columns(path: Path) -> list[str]:
    """Read CSV header names if the file exists."""
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        return list(reader.fieldnames or [])


def _read_json_top_keys(path: Path) -> list[str]:
    """Read top-level JSON object keys if present and object-like."""
    if not path.exists():
        return []
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict):
        obj = cast(dict[str, Any], data)
        return sorted(obj)
    return []


toy_offtarget = toy_dir / "sirnaforge" / "zfn_offtarget_sites.csv"
real_offtarget = real_dir / "sirnaforge" / "zfn_offtarget_sites.csv"

toy_summary = toy_dir / "logs" / "workflow_summary.json"
real_summary = real_dir / "logs" / "workflow_summary.json"

toy_candidate = toy_dir / "sirnaforge" / "zfn_candidate_summary.json"
real_candidate = real_dir / "sirnaforge" / "zfn_candidate_summary.json"

rows: list[dict[str, Any]] = []
for label, toy_file, real_file in [
    ("offtarget_csv_columns", toy_offtarget, real_offtarget),
    ("candidate_json_keys", toy_candidate, real_candidate),
    ("workflow_summary_keys", toy_summary, real_summary),
]:
    if label == "offtarget_csv_columns":
        toy_keys = _read_csv_columns(toy_file)
        real_keys = _read_csv_columns(real_file)
    else:
        toy_keys = _read_json_top_keys(toy_file)
        real_keys = _read_json_top_keys(real_file)

    rows.append(
        {
            "artifact": label,
            "toy_exists": toy_file.exists(),
            "real_exists": real_file.exists(),
            "toy_key_count": len(toy_keys),
            "real_key_count": len(real_keys),
            "schemas_match": toy_keys == real_keys,
            "only_in_toy": sorted(set(toy_keys) - set(real_keys)),
            "only_in_real": sorted(set(real_keys) - set(toy_keys)),
        }
    )

parity_df = pd.DataFrame(rows)
parity_df